# 15장. 하나의 데이터 분석 프로젝트로 완성하기

이 노트북은 `book/chapters/ch15_final_project.md` 강의안을 초보자가 그대로 따라 하며 이해할 수 있도록 구성한 기말 종합 프로젝트 실습 자료입니다.

이번 장의 핵심은 온라인 쇼핑몰 데이터를 바탕으로 **EDA + 시각화 + 머신러닝 + LLM 활용 기록 + 외부 데이터 통합 + 자동화 설계**를 하나의 분석 프로젝트로 연결하는 것입니다.


## 0. 이 노트북 사용 방법

아래 셀을 위에서부터 차례대로 실행하세요.

- 원본 데이터는 `data/raw/` 폴더의 `customers.csv`, `products.csv`, `orders.csv`, `order_items.csv`를 사용합니다.
- 데이터가 없다면 먼저 `python scripts/generate_sample_data.py`를 실행하세요.
- 프로젝트 산출물은 `reports/`와 `reports/figures/`에 저장됩니다.
- 외부 데이터는 예시 공휴일 데이터 `data/external/holidays.csv`를 사용합니다.
- 마지막에는 `python scripts/run_final_project.py`로 전체 프로젝트를 한 번에 실행할 수 있습니다.


## 1. 최종 프로젝트의 전체 흐름

최종 프로젝트는 하나의 분석 스토리를 만드는 과정입니다. 코드, 그래프, 모델, 보고서가 따로 존재하는 것이 아니라 하나의 질문을 향해 연결되어야 합니다.

```text
분석 목적 정의
→ 데이터 구조 점검
→ 전처리 기준 수립
→ EDA와 핵심 지표 계산
→ 시각화
→ 회귀 또는 분류 모델링
→ LLM 활용과 검증 기록
→ 외부 데이터 선택 수집 또는 연결
→ 외부 데이터 통합 분석
→ 자동화/파이프라인 설계
→ 최종 보고서와 발표 정리
```


## 2. 프로젝트 산출물

기말 프로젝트에서는 파일을 많이 만드는 것보다 각 산출물이 분석 흐름의 어느 단계를 설명하는지 명확해야 합니다.

| 구분 | 예시 파일 | 의미 |
|---|---|---|
| 데이터 개요 | `reports/ch15_dataset_summary.csv` | 사용 데이터의 구조 요약 |
| 전처리 비교 | `reports/ch15_preprocessing_comparison.csv` | 전처리 전후 행/열 수 비교 |
| EDA 결과 | `reports/ch15_category_sales.csv`, `reports/ch15_monthly_sales.csv` | 주요 분석 질문에 대한 집계 결과 |
| 시각화 | `reports/figures/ch15_*.png` | 분석 결과를 그래프로 전달 |
| 머신러닝 결과 | `reports/ch15_regression_model_comparison.csv`, `reports/ch15_classification_model_comparison.csv` | 회귀/분류 모델 결과 |
| 외부 데이터 결과 | `reports/ch15_holiday_sales_comparison.csv` | 공휴일 데이터와 매출 데이터 연결 결과 |
| LLM 기록 | `reports/ch15_llm_usage_log.md` | LLM 활용 목적과 검증 내역 |
| 자동화 설계 | `reports/ch15_automation_plan.md` | 반복 분석 운영화 설계 |
| 최종 보고서 | `reports/ch15_final_report.md` | 분석 결과, 해석, 한계, 다음 단계 정리 |


## 3. 패키지와 경로 설정

프로젝트 루트, 데이터 폴더, 외부 데이터 폴더, 보고서 폴더를 설정합니다.


In [ ]:
from pathlib import Path
import sys

import pandas as pd

CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == 'notebooks':
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
EXTERNAL_DIR = PROJECT_ROOT / 'data' / 'external'
REPORT_DIR = PROJECT_ROOT / 'reports'
FIGURE_DIR = REPORT_DIR / 'figures'

for path in [PROCESSED_DIR, EXTERNAL_DIR, REPORT_DIR, FIGURE_DIR]:
    path.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('프로젝트 루트:', PROJECT_ROOT)
print('원본 데이터 폴더:', RAW_DIR)
print('전처리 데이터 폴더:', PROCESSED_DIR)
print('외부 데이터 폴더:', EXTERNAL_DIR)
print('보고서 폴더:', REPORT_DIR)
print('그림 폴더:', FIGURE_DIR)


## 4. 기말 프로젝트 공통 함수 불러오기

15장 프로젝트 흐름은 `src/final_project.py`에 함수로 정리했습니다. 노트북에서는 각 단계를 직접 실행하면서 결과를 확인합니다.


In [ ]:
from src.final_project import (
    analyze_holiday_sales,
    build_automation_plan,
    build_dataset_summary,
    build_final_report,
    build_insight_cards,
    build_llm_usage_log,
    build_project_tables,
    generate_project_figures,
    load_raw_data,
    preprocess_project_data,
    run_final_project,
    save_processed_data,
    save_project_tables,
    train_classification_models,
    train_regression_models,
)


## 5. 데이터 불러오기와 구조 점검

먼저 원본 데이터를 불러오고, 최종 보고서에 사용할 데이터 구조 요약표를 만듭니다.


In [ ]:
raw_data = load_raw_data(PROJECT_ROOT)
dataset_summary = build_dataset_summary(raw_data)

dataset_summary.to_csv(REPORT_DIR / 'ch15_dataset_summary.csv', index=False, encoding='utf-8-sig')
dataset_summary


## 6. 전처리 기준 수립과 저장

전처리는 분석 목적에 맞게 데이터를 사용할 수 있는 상태로 만드는 과정입니다. 문자열 공백, 숫자형, 날짜, 주문 상태, `line_total`을 정리하고 전처리 전후 비교표를 저장합니다.


In [ ]:
processed, preprocessing_comparison = preprocess_project_data(raw_data)
processed_outputs = save_processed_data(processed, PROJECT_ROOT)

preprocessing_comparison.to_csv(REPORT_DIR / 'ch15_preprocessing_comparison.csv', index=False, encoding='utf-8-sig')
display(preprocessing_comparison)
processed_outputs


## 7. EDA와 핵심 지표 계산

주문 상세, 상품, 주문, 고객 데이터를 연결한 뒤 카테고리별 매출, 월별 매출, 고객별 구매 금액, 상품별 매출을 계산합니다.


In [ ]:
tables = build_project_tables(processed)
table_outputs = save_project_tables(tables, PROJECT_ROOT)

display(tables['merge_validation'])
display(tables['category_sales'].head())
display(tables['monthly_sales'].head())
display(tables['customer_sales'].head())
display(tables['product_sales'].head())
table_outputs


## 8. 시각화 결과 저장

카테고리별 매출, 월별 매출 추이, 구매 금액 상위 고객, 상품 가격과 판매 수량 관계 그래프를 저장합니다.

그래프 파일은 `reports/figures/ch15_*.png`에 저장됩니다.


In [ ]:
figure_outputs = generate_project_figures(tables, PROJECT_ROOT)
figure_outputs


## 9. 머신러닝 모델 추가

최종 프로젝트에는 간단한 머신러닝 모델을 하나 이상 포함할 수 있습니다. 여기서는 회귀 모델로 주문별 총금액을 예측하고, 분류 모델로 주문 취소 여부를 예측합니다.

주의: 분류 모델에서는 `order_status`를 입력값으로 사용하지 않습니다. 정답을 만드는 컬럼이기 때문에 데이터 누수가 발생합니다.


In [ ]:
regression_comparison = train_regression_models(processed)
classification_comparison = train_classification_models(processed)

regression_comparison.to_csv(REPORT_DIR / 'ch15_regression_model_comparison.csv', index=False, encoding='utf-8-sig')
classification_comparison.to_csv(REPORT_DIR / 'ch15_classification_model_comparison.csv', index=False, encoding='utf-8-sig')

display(regression_comparison)
display(classification_comparison)


## 10. 외부 데이터 통합: 공휴일 예시

13장에서 다룬 외부 데이터 수집 내용을 기말 프로젝트 안에 연결합니다. 여기서는 예시 공휴일 데이터 `data/external/holidays.csv`를 만들고, 일자별 매출과 연결해 공휴일 여부별 평균 매출을 비교합니다.

이 결과는 공휴일과 매출 차이가 함께 관찰되는지를 보는 것이며, 공휴일이 매출 변화의 원인이라고 단정할 수 없습니다.


In [ ]:
holiday_result = analyze_holiday_sales(tables, PROJECT_ROOT)

display(holiday_result['holiday_summary'])
display(holiday_result['holiday_sales_comparison'])
holiday_result['external_integration_path']


## 11. 인사이트 카드 만들기

분석 결과는 보고서에 바로 넣을 수 있는 형태로 정리합니다. 관찰, 주의점, 다음 단계를 분리해 과장 해석을 줄입니다.


In [ ]:
insight_cards = build_insight_cards(
    tables=tables,
    regression=regression_comparison,
    holiday_comparison=holiday_result['holiday_sales_comparison'],
)

insight_cards.to_csv(REPORT_DIR / 'ch15_insight_cards.csv', index=False, encoding='utf-8-sig')
insight_cards


## 12. LLM 활용 및 검증 기록

LLM은 분석 질문 보완, 코드 초안 작성, 오류 해결, 해석 문장 작성, 보고서 검토에 활용할 수 있습니다. 다만 최종 산출물에는 검증된 내용만 반영해야 합니다.


In [ ]:
llm_usage_log = build_llm_usage_log()
llm_usage_log.to_csv(REPORT_DIR / 'ch15_llm_usage_log.csv', index=False, encoding='utf-8-sig')

llm_usage_text = f'''# Chapter 15 LLM 활용 및 검증 기록

## 1. LLM 활용 목적

기말 종합 프로젝트에서 LLM은 분석 질문 보완, pandas 코드 초안 작성, 머신러닝 코드 검토, 외부 데이터 연결 검토, 오류 해결, 해석 문장 작성, 자동화 설계 보조 도구로 사용했습니다.

## 2. LLM 활용 기록

```text
{llm_usage_log.to_string(index=False)}
```

## 3. 검증 원칙

- 원본 개인정보나 개별 주문 데이터를 LLM에 입력하지 않았습니다.
- 컬럼명, 데이터 구조, 집계 결과 중심으로 질문했습니다.
- LLM이 생성한 코드는 실제 데이터로 실행해 검증했습니다.
- 머신러닝 코드에서는 데이터 누수 여부를 확인했습니다.
- 외부 데이터 연결에서는 출처, 연결 키, 병합 전후 행 수를 확인했습니다.
- LLM이 작성한 해석 문장은 원인 단정 여부를 검토했습니다.
- 최종 보고서에는 검증된 코드와 문장만 반영했습니다.
'''

llm_usage_path = REPORT_DIR / 'ch15_llm_usage_log.md'
llm_usage_path.write_text(llm_usage_text, encoding='utf-8')
display(llm_usage_log)
llm_usage_path


## 13. 자동화와 파이프라인 설계

최종 프로젝트의 마지막 단계는 반복 가능한 분석 흐름을 설계하는 것입니다. Airflow는 분석 파이프라인 실행 순서와 로그 관리에, Make/n8n은 보고서 전달과 외부 서비스 연결에 적합합니다.


In [ ]:
automation_plan = build_automation_plan()
automation_plan_path = REPORT_DIR / 'ch15_automation_plan.md'
automation_plan_path.write_text(automation_plan, encoding='utf-8')
print(automation_plan)


## 14. 최종 보고서 작성

최종 보고서는 분석 결과를 단순히 나열하는 문서가 아니라, 분석 목적, 데이터, 처리 과정, 주요 결과, 해석, 한계를 연결해서 보여 주어야 합니다.


In [ ]:
final_report = build_final_report(
    dataset_summary=dataset_summary,
    preprocessing_comparison=preprocessing_comparison,
    tables=tables,
    regression=regression_comparison,
    classification=classification_comparison,
    holiday_comparison=holiday_result['holiday_sales_comparison'],
    insight_cards=insight_cards,
)

final_report_path = REPORT_DIR / 'ch15_final_report.md'
final_report_path.write_text(final_report, encoding='utf-8')
print('최종 보고서 저장 완료:', final_report_path)


## 15. 전체 프로젝트를 한 번에 실행하기

위에서 단계별로 실행한 전체 프로젝트는 `run_final_project()` 함수로 한 번에 실행할 수 있습니다. 터미널에서는 아래 명령을 사용합니다.

```bash
python scripts/run_final_project.py
```


In [ ]:
final_project_result = run_final_project(PROJECT_ROOT)

display(final_project_result['deliverables'])
final_project_result['final_report_path']


## 16. 최종 제출 전 점검표

최종 제출 전 아래 항목을 확인하세요.

| 점검 항목 | 확인 |
|---|---|
| 분석 목적이 명확한가? | □ |
| 데이터 구조 요약표가 있는가? | □ |
| 전처리 기준과 전후 비교가 있는가? | □ |
| EDA 결과와 시각화가 같은 질문을 향하는가? | □ |
| 회귀 또는 분류 모델 결과가 있는가? | □ |
| 데이터 누수와 평가 지표를 검토했는가? | □ |
| 외부 데이터 출처와 연결 기준을 기록했는가? | □ |
| LLM 활용 및 검증 기록이 있는가? | □ |
| 자동화 설계가 포함되어 있는가? | □ |
| 최종 보고서가 원인 단정을 피하고 한계를 설명하는가? | □ |
| 산출물 파일 경로가 정리되어 있는가? | □ |


## 17. 실습 과제

아래 과제를 직접 해결해 보세요.

1. `reports/ch15_final_report.md`를 열어 보고서 문장이 자연스럽게 이어지는지 확인하세요.
2. 인사이트 카드 중 하나를 골라 그래프와 연결되는지 설명하세요.
3. 회귀와 분류 중 본인 프로젝트에 더 적합한 모델 하나를 선택하고 이유를 적으세요.
4. 공휴일 데이터 대신 다른 외부 데이터를 연결한다면 어떤 키로 연결할지 정리하세요.
5. LLM 활용 기록에서 실제 사용하지 않은 항목은 수정하거나 삭제하세요.
6. 자동화 설계에서 Airflow가 담당할 부분과 Make/n8n이 담당할 부분을 다시 나누어 보세요.


In [ ]:
# 과제 1. 최종 프로젝트 산출물 목록을 다시 확인해 보세요.
deliverables_path = REPORT_DIR / 'ch15_project_deliverables.csv'
if deliverables_path.exists():
    deliverables = pd.read_csv(deliverables_path)
    display(deliverables)
else:
    print('아직 산출물 목록 파일이 없습니다. run_final_project(PROJECT_ROOT)를 먼저 실행하세요.')


## 18. 정리

이번 장에서는 다음 내용을 하나의 프로젝트로 연결했습니다.

- 분석 목적 정의와 프로젝트 흐름 설계
- 데이터 구조 점검과 전처리 기준 수립
- 카테고리별, 월별, 고객별, 상품별 EDA
- 주요 시각화 결과 저장
- 회귀 모델과 분류 모델 결과 비교
- 공휴일 외부 데이터와 일자별 매출 연결
- 인사이트 카드 작성
- LLM 활용 및 검증 기록 작성
- 자동화 설계서 작성
- 최종 보고서와 산출물 목록 생성
- `src/final_project.py`와 `scripts/run_final_project.py`로 재현 가능한 프로젝트 구조 만들기

이제 데이터 분석의 기본 흐름부터 LLM 활용, 외부 데이터, 자동화 설계까지 하나의 최종 프로젝트로 정리할 수 있습니다.
